# Predicción de Abandono de Clientes (Telco Customer Churn)
## Notebook 2 de 2 Modelamiento supervisado, no supervisado y evaluación

**MLY1101 Machine Learning | Caso A | Evaluación Parcial N°2**

Continúa el notebook `01_EDA_Preparacion_Churn.ipynb` y desarrolla las fases 4 y 5 de CRISP-DM (Modelamiento y Evaluación) y una propuesta de Despliegue (fase 6).

## 1. Flujo de trabajo (CRISP-DM)

1. **Negocio y datos** (notebook 1): problema, KPIs, origen y calidad de datos, EDA.
2. **Preparación** (notebook 1 + Pipeline aquí): dataset limpio y codificado; el escalado se aplica **dentro** de cada `Pipeline`, ajustándose solo con datos de entrenamiento.
3. **Modelamiento**: línea base → Regresión Logística → Random Forest, con búsqueda de hiperparámetros por validación cruzada estratificada (5 particiones) **solo sobre train**.
4. **Evaluación**: el conjunto de prueba (20%) se usa una sola vez al final; el umbral de decisión se elige con predicciones fuera de muestra de train (no con test).
5. **No supervisado**: segmentación K-Means para diseñar campañas por perfil.
6. **Despliegue** (propuesta): Sección 10.

## 2. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import joblib

from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve, silhouette_score, adjusted_rand_score)

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold"})
RANDOM_STATE = 42
COLOR_NO, COLOR_SI = "#2a78d6", "#eb6834"
C_LR, C_RF = "#2a78d6", "#1baf7a"
SEG_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
IMG, MODELS = "../images/", "../models/"

In [ ]:
df = pd.read_csv("../data/telco_churn_limpio.csv")   # generado por el notebook 1
y = df["Churn"]
X_all = df.drop(columns=["Churn"])

# Misma partición que el notebook 1 (80/20 estratificada, misma semilla)
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Train: {len(X_train_all)} ({y_train.mean():.1%} churn) | Test: {len(X_test_all)} ({y_test.mean():.1%} churn)")

## 3. Selección de variables: ¿se usan los atributos demográficos protegidos?

El EDA mostró que `gender` no se asocia al churn y que `SeniorCitizen` sí, pero en parte porque los adultos mayores contratan planes más caros. Antes de modelar se prueba con validación cruzada (configuración fija, solo train) si excluir estas variables cuesta desempeño. Si no cuesta, se excluyen: así el modelo no puede discriminar directamente por género o edad, y ambas variables quedan solo para **auditar** el resultado (Sección 8).

In [ ]:
NUM_MINMAX, NUM_ROBUST = ["tenure", "MonthlyCharges"], ["TotalCharges"]

def preprocesador():
    # Escalado elegido en el notebook 1 según la forma de cada variable; el resto (0/1) pasa sin cambios
    return ColumnTransformer([("minmax", MinMaxScaler(), NUM_MINMAX),
                              ("robust", RobustScaler(), NUM_ROBUST)], remainder="passthrough")

conjuntos = {
    "Todas las variables": list(X_all.columns),
    "Sin gender": [c for c in X_all.columns if c != "gender_Female"],
    "Sin gender ni SeniorCitizen": [c for c in X_all.columns if c not in ("gender_Female", "SeniorCitizen")],
}
METRICAS_CV = ["recall", "precision", "f1", "roc_auc", "average_precision"]
filas = []
for nombre, cols in conjuntos.items():
    for mod_nombre, modelo in [
        ("Reg. Logística", LogisticRegression(class_weight="balanced", max_iter=2000, solver="liblinear")),
        ("Random Forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight="balanced",
                                                 random_state=RANDOM_STATE, n_jobs=-1))]:
        pipe = Pipeline([("prep", preprocesador()), ("modelo", modelo)])
        r = cross_validate(pipe, X_train_all[cols], y_train, cv=cv, scoring=METRICAS_CV)
        filas.append([nombre, mod_nombre, len(cols)] + [r[f"test_{m}"].mean() for m in METRICAS_CV])
comp_vars = pd.DataFrame(filas, columns=["Conjunto", "Modelo", "n_variables"] + METRICAS_CV).round(3)
comp_vars

**Observación:** excluir `gender` y `SeniorCitizen` **no cambia el desempeño**: la Regresión Logística mantiene F1 = 0,629 y PR-AUC = 0,659 en los tres conjuntos, y el Random Forest pasa de F1 0,635 a 0,633 y de PR-AUC 0,662 a 0,664. Las diferencias (≤ 0,002) son menores que la variación entre particiones. **Decisión:** se entrenan los modelos **sin** estas dos variables. Se gana en ética (el modelo no puede usar directamente género ni edad) sin perder capacidad predictiva, y ambas se guardan solo para la auditoría de sesgo.

In [ ]:
FEATURES = conjuntos["Sin gender ni SeniorCitizen"]
X_train, X_test = X_train_all[FEATURES], X_test_all[FEATURES]
print(f"{len(FEATURES)} variables predictoras:", FEATURES)

## 4. Línea base (modelo trivial)

Un modelo que siempre predice la clase mayoritaria ("No abandona") fija el piso que cualquier modelo útil debe superar.

In [ ]:
def metricas(y_true, y_pred, y_score=None):
    m = {"Accuracy": accuracy_score(y_true, y_pred),
         "Recall": recall_score(y_true, y_pred, zero_division=0),
         "Precisión": precision_score(y_true, y_pred, zero_division=0),
         "F1": f1_score(y_true, y_pred, zero_division=0)}
    if y_score is not None:
        m["ROC-AUC"] = roc_auc_score(y_true, y_score)
        m["PR-AUC"] = average_precision_score(y_true, y_score)
    return m

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
res_base = metricas(y_test, base.predict(X_test), base.predict_proba(X_test)[:, 1])
pd.Series(res_base).round(3)

**Observación:** la línea base obtiene **73,5% de accuracy** sin detectar a **ningún** cliente que se va (recall 0). Confirma que la accuracy es engañosa con clases desbalanceadas; los modelos se comparan por **recall, precisión, F1 y PR-AUC** (área bajo la curva precisión-recall, que se centra en la clase minoritaria; su piso es la tasa de churn, 0,265).

## 5. Modelos supervisados

Problema de **clasificación binaria** (abandona / no abandona) con probabilidad de salida, para poder ordenar clientes por riesgo.

### 5.1 Modelo 1: Regresión Logística

**Por qué este algoritmo:**
- Modela directamente la **probabilidad** de churn y sus coeficientes son **interpretables** (cuánto sube o baja el riesgo cada característica), algo necesario para explicarle al área comercial por qué un cliente está en riesgo.
- Es el modelo con mejor recall en los estudios previos sobre este mismo dataset (Zhu 2023; Wanda 2024; ver notebook 1).
- Es sensible a la escala de las variables ⇒ usa el escalado justificado en el notebook 1 (MinMax / Robust).

**Configuración:**
- `class_weight="balanced"`: penaliza más los errores en la clase minoritaria (churn), compensando el desbalance 1 : 2,8 sin generar datos sintéticos.
- Hiperparámetros buscados: `C` (inverso de la regularización: valores bajos = modelo más simple, controla el sobreajuste y la colinealidad entre `tenure` y `TotalCharges`) y `penalty` (`l2` reparte el peso entre variables correlacionadas; `l1` puede dejar coeficientes en cero y seleccionar variables).
- Criterio de búsqueda: **F1** (equilibrio entre recall y precisión; optimizar solo recall llevaría a marcar a todos como riesgo).

In [ ]:
pipe_lr = Pipeline([("prep", preprocesador()),
                    ("modelo", LogisticRegression(class_weight="balanced", max_iter=5000, solver="liblinear"))])
grid_lr = GridSearchCV(pipe_lr, {"modelo__C": [0.01, 0.1, 1, 10], "modelo__penalty": ["l1", "l2"]},
                       scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_lr.best_params_, "| F1 CV:", round(grid_lr.best_score_, 4))
pd.DataFrame(grid_lr.cv_results_)[["param_modelo__C", "param_modelo__penalty", "mean_test_score", "std_test_score"]] \
  .sort_values("mean_test_score", ascending=False).round(4).head(8)

**Observación:** la mejor configuración es `C = 1` con penalización `l1` (F1 en validación 0,631). Las 8 combinaciones probadas quedan entre 0,624 y 0,631, una diferencia menor que la desviación entre particiones (≈ 0,02): el modelo es **poco sensible** a la regularización, señal de que no está sobreajustando. La penalización `l1` dejó en cero el coeficiente de `MonthlyCharges` (ver Sección 7): su información ya está contenida en los servicios contratados (fibra, streaming, etc.), que son los que determinan el cargo.

### 5.2 Modelo 2: Random Forest

**Por qué este algoritmo:**
- Captura **relaciones no lineales e interacciones** sin especificarlas. El EDA mostró una interacción clara: el cargo mensual alto aumenta el churn solo cuando va con contrato mes a mes (deciles, notebook 1). Una regresión logística no la modela salvo que se agregue a mano.
- Es un ensamble de árboles con *bagging*: promedia muchos árboles entrenados con muestras distintas, lo que reduce la varianza y el sobreajuste de un árbol individual.
- Entrega importancia de variables para interpretar.
- No necesita escalado; se mantiene el mismo `Pipeline` para comparar ambos modelos en igualdad de condiciones (el escalado no altera los cortes de los árboles).

**Configuración:**
- `n_estimators=300` árboles (suficiente para estabilizar el promedio), `class_weight="balanced"`.
- Hiperparámetros buscados: `max_depth` (profundidad máxima), `min_samples_leaf` (mínimo de clientes por hoja; valores altos evitan reglas basadas en pocos clientes) y `max_features` (variables candidatas en cada corte; menos variables = árboles más distintos entre sí).

In [ ]:
pipe_rf = Pipeline([("prep", preprocesador()),
                    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                      random_state=RANDOM_STATE, n_jobs=-1))])
grid_rf = GridSearchCV(pipe_rf, {"modelo__max_depth": [None, 6, 10],
                                 "modelo__min_samples_leaf": [1, 5, 10, 20],
                                 "modelo__max_features": ["sqrt", 0.5]},
                       scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_rf.best_params_, "| F1 CV:", round(grid_rf.best_score_, 4))
pd.DataFrame(grid_rf.cv_results_)[["param_modelo__max_depth", "param_modelo__min_samples_leaf",
                                   "param_modelo__max_features", "mean_test_score", "std_test_score"]] \
  .sort_values("mean_test_score", ascending=False).round(4).head(8)

In [ ]:
# Sobreajuste: F1 en train vs F1 en validación cruzada para distintos min_samples_leaf (resto en su mejor valor)
cvr = pd.DataFrame(GridSearchCV(pipe_rf, {"modelo__max_depth": [grid_rf.best_params_["modelo__max_depth"]],
                                          "modelo__min_samples_leaf": [1, 5, 10, 20],
                                          "modelo__max_features": [grid_rf.best_params_["modelo__max_features"]]},
                                scoring="f1", cv=cv, n_jobs=-1, return_train_score=True).fit(X_train, y_train).cv_results_)
cvr[["param_modelo__min_samples_leaf", "mean_train_score", "mean_test_score"]].round(3) \
   .rename(columns={"param_modelo__min_samples_leaf": "min_samples_leaf", "mean_train_score": "F1 train",
                    "mean_test_score": "F1 validación"})

**Observación:** la mejor configuración es `max_depth = 10`, `max_features = 0.5` y `min_samples_leaf = 10` (F1 en validación 0,637). La tabla de sobreajuste justifica el `min_samples_leaf`: con hojas de 1 cliente el bosque memoriza train (F1 train 0,809 vs validación 0,629, brecha de 0,18); con hojas de 10 clientes la brecha baja a 0,07 (0,708 vs 0,637) y el F1 de validación es el más alto. Limitar la profundidad y el tamaño de hoja **controla el sobreajuste**.

### 5.3 Comparación en validación cruzada (estabilidad)

Se evalúan ambos modelos ya ajustados con las mismas 5 particiones de train. Se reporta media y desviación estándar: un modelo con menor desviación es más estable ante cambios en los datos.

In [ ]:
modelos = {"Regresión Logística": grid_lr.best_estimator_, "Random Forest": grid_rf.best_estimator_}
cv_res = {n: cross_validate(m, X_train, y_train, cv=cv, scoring=METRICAS_CV) for n, m in modelos.items()}
tabla_cv = pd.DataFrame({n: {m: f"{r['test_'+m].mean():.3f} ± {r['test_'+m].std():.3f}" for m in METRICAS_CV}
                         for n, r in cv_res.items()}).T
tabla_cv

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, m, et in zip(axes, ["recall", "precision", "f1", "average_precision"], ["Recall", "Precisión", "F1", "PR-AUC"]):
    datos = [cv_res[n][f"test_{m}"] for n in modelos]
    bp = ax.boxplot(datos, tick_labels=["Reg. Log.", "Random F."], patch_artist=True, widths=0.5,
                    medianprops={"color": "black"})
    for p, c in zip(bp["boxes"], [C_LR, C_RF]): p.set_facecolor(c); p.set_alpha(0.7)
    ax.set_title(et)
fig.suptitle("Gráfico 6. Métricas en las 5 particiones de validación cruzada (train)", fontweight="bold")
plt.tight_layout(); plt.savefig(IMG + "g6_cv_comparacion.png", dpi=130); plt.show()

**Observación:** en validación cruzada los modelos quedan prácticamente empatados y con perfiles distintos:
- **Regresión Logística:** más **recall** (0,803 ± 0,039): detecta más clientes que se van.
- **Random Forest:** más **precisión** (0,541 vs 0,520) y algo más de F1 (0,637 vs 0,631) y PR-AUC (0,664 vs 0,660). Además es **más estable**: su desviación en recall (± 0,021) es la mitad que la de la Regresión Logística (± 0,039).
- ROC-AUC prácticamente igual (0,846 vs 0,845).

Todas las diferencias entre medias son menores que la desviación entre particiones: ningún modelo es claramente superior.

## 6. Evaluación en el conjunto de prueba

### 6.1 Métricas con umbral estándar (0,5)
El conjunto de prueba (1.409 clientes no vistos) se usa aquí por primera vez.

In [ ]:
prob_test = {n: m.predict_proba(X_test)[:, 1] for n, m in modelos.items()}
tabla_test = pd.DataFrame({"Línea base": res_base,
                           **{n: metricas(y_test, (p >= 0.5).astype(int), p) for n, p in prob_test.items()}}).T
tabla_test.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (n, p) in zip(axes, prob_test.items()):
    cm = confusion_matrix(y_test, (p >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["No abandona", "Abandona"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{n} (umbral 0,5)\nDetecta {tp} de {tp+fn} que se van; {fp} falsas alarmas")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Valor real"); ax.grid(False)
plt.tight_layout(); plt.savefig(IMG + "g7_matrices_confusion.png", dpi=130); plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
for (n, p), c in zip(prob_test.items(), [C_LR, C_RF]):
    fpr, tpr, _ = roc_curve(y_test, p)
    a1.plot(fpr, tpr, color=c, lw=2, label=f"{n} (AUC = {roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    a2.plot(rc, pr, color=c, lw=2, label=f"{n} (PR-AUC = {average_precision_score(y_test, p):.3f})")
a1.plot([0, 1], [0, 1], "k--", lw=1, label="Azar (AUC = 0,5)")
a1.set_xlabel("Tasa de falsos positivos"); a1.set_ylabel("Tasa de verdaderos positivos (recall)")
a1.set_title("Curva ROC"); a1.legend(loc="lower right")
a2.axhline(y_test.mean(), color="black", ls="--", lw=1, label=f"Azar (= tasa de churn {y_test.mean():.3f})")
a2.set_xlabel("Recall"); a2.set_ylabel("Precisión"); a2.set_title("Curva Precisión-Recall"); a2.legend(loc="upper right")
fig.suptitle("Gráfico 8. Capacidad de discriminación en test", fontweight="bold")
plt.tight_layout(); plt.savefig(IMG + "g8_curvas_roc_pr.png", dpi=130); plt.show()

**Observación:** los resultados en test son coherentes con la validación cruzada (no hay señales de sobreajuste):
- Ambos modelos **detectan 295 de los 374 clientes que se van** (recall 0,789).
- El Random Forest genera **34 falsas alarmas menos** (258 vs 292), por eso tiene más precisión (0,533 vs 0,503), F1 (0,636 vs 0,614) y accuracy (0,761 vs 0,737).
- Ambos superan ampliamente la línea base: PR-AUC 0,634 y 0,649 vs 0,265; ROC-AUC 0,841 y 0,843 vs 0,5.
- Los resultados están en el rango reportado por Zhu (2023) para la Regresión Logística con este dataset (recall 0,84, F1 0,65, AUC 0,78).

### 6.2 Elección del modelo final

**Criterio definido antes de mirar test:** se elige el modelo con mayor **PR-AUC en validación cruzada** (mide qué tan bien ordena a los clientes por riesgo, independiente del umbral, y se centra en la clase churn). Si la diferencia es menor que la desviación estándar entre particiones, se considera empate y se prefiere el modelo **más simple e interpretable**.

In [ ]:
ap = {n: (r["test_average_precision"].mean(), r["test_average_precision"].std()) for n, r in cv_res.items()}
dif = ap["Random Forest"][0] - ap["Regresión Logística"][0]
desv = max(ap["Random Forest"][1], ap["Regresión Logística"][1])
print(f"PR-AUC CV  RL = {ap['Regresión Logística'][0]:.4f} ± {ap['Regresión Logística'][1]:.4f} | "
      f"RF = {ap['Random Forest'][0]:.4f} ± {ap['Random Forest'][1]:.4f}")
print(f"Diferencia RF - RL = {dif:.4f} | desviación de referencia = {desv:.4f}")
MODELO_FINAL = "Random Forest" if dif > desv else "Regresión Logística"
print("Modelo final:", MODELO_FINAL)
modelo_final = modelos[MODELO_FINAL]

**Decisión: Regresión Logística.** La diferencia en PR-AUC (0,0045) es mucho menor que la desviación entre particiones (0,0205): es un **empate técnico**, y en ese caso se elige el modelo más simple e interpretable. La Regresión Logística permite explicarle al área comercial **por qué** un cliente está en riesgo (coeficientes y *odds ratio*, Sección 7) y coincide con la elección de los estudios previos.

**Costo de la decisión (declarado):** en test, el Random Forest logró el mismo recall con 34 falsas alarmas menos. No se cambia la elección por eso: usar test para elegir modelo haría que el resultado reportado sea optimista. El Random Forest queda guardado como **modelo alternativo** para compararlo en un piloto real (Sección 10).

### 6.3 Ajuste del umbral de decisión

El umbral 0,5 no es obligatorio: bajarlo detecta más clientes que se van (más recall) a cambio de más falsas alarmas (menos precisión). Para elegirlo **sin usar test**, se generan probabilidades *fuera de muestra* sobre train con validación cruzada (`cross_val_predict`).

**Regla de negocio:** detectar al menos **80% de los clientes que se van** (meta de recall, en línea con lo reportado en estudios previos) y, entre los umbrales que lo cumplen, tomar el **más alto** (el que genera menos falsas alarmas).

In [ ]:
META_RECALL = 0.80
prob_oof = cross_val_predict(modelo_final, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
umbrales = np.round(np.arange(0.20, 0.71, 0.01), 2)
curva = pd.DataFrame([{"umbral": u, **metricas(y_train, (prob_oof >= u).astype(int)),
                       "% contactados": (prob_oof >= u).mean()} for u in umbrales])
UMBRAL = curva.loc[curva["Recall"] >= META_RECALL, "umbral"].max()
print(f"Umbral elegido: {UMBRAL:.2f}")
curva[curva["umbral"].isin([0.30, 0.35, 0.40, 0.45, UMBRAL, 0.50, 0.55, 0.60])].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for m, c in [("Recall", COLOR_SI), ("Precisión", COLOR_NO), ("F1", "#555555")]:
    ax.plot(curva["umbral"], curva[m], color=c, lw=2, label=m)
ax.plot(curva["umbral"], curva["% contactados"], color="#999999", lw=1.5, ls=":", label="% de clientes contactados")
ax.axvline(UMBRAL, color="black", ls="--", lw=1)
ax.text(UMBRAL + 0.005, 0.95, f"umbral elegido = {UMBRAL:.2f}", fontsize=10)
ax.axhline(META_RECALL, color=COLOR_SI, ls=":", lw=1)
ax.set_xlabel("Umbral de probabilidad para marcar 'riesgo de churn'"); ax.set_ylabel("Valor")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title(f"Gráfico 9. Efecto del umbral ({MODELO_FINAL}, predicciones fuera de muestra en train)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False)
plt.tight_layout(); plt.savefig(IMG + "g9_umbral.png", dpi=130); plt.show()

In [ ]:
pred_final = (prob_test[MODELO_FINAL] >= UMBRAL).astype(int)
res_final = metricas(y_test, pred_final, prob_test[MODELO_FINAL])
# Referencia: cómo se ve el test con umbrales vecinos (solo informativo; el umbral ya se eligió sin test)
vecinos = pd.DataFrame([{"umbral": u, **metricas(y_test, (prob_test[MODELO_FINAL] >= u).astype(int)),
                         "contactados": int((prob_test[MODELO_FINAL] >= u).sum())} for u in [0.40, 0.45, 0.50, 0.55, 0.60]])
vecinos["elegido"] = np.where(np.isclose(vecinos["umbral"], UMBRAL), "◀", "")
display(vecinos.round(3))
cm = confusion_matrix(y_test, pred_final); tn, fp, fn, tp = cm.ravel()
print(f"Con umbral {UMBRAL:.2f}: detecta {tp} de {tp+fn} clientes que se van; {fp} falsas alarmas; "
      f"{fn} no detectados; contacta a {tp+fp} de {len(y_test)} clientes ({(tp+fp)/len(y_test):.1%}).")

**Observación:** la regla de negocio da un umbral de **0,50**: es el umbral más alto con el que, fuera de muestra, se detecta al menos 80% de los que se van (recall 0,803). Bajar el umbral sube el recall pero obliga a contactar a más clientes: con 0,45 el recall sube a 0,833 y se contacta a 44,9% de la base (vs 41,0%); con 0,40, a 0,864 y 49,0%.

En test, con umbral 0,50 el modelo detecta **295 de 374 clientes que se van (78,9%)**, genera 292 falsas alarmas, deja 79 sin detectar y pide contactar a **587 de 1.409 clientes (41,7%)**. El recall en test (78,9%) queda levemente bajo la meta de 80%, dentro de la variación esperada (la desviación del recall en validación cruzada fue ± 3,9 pp). Si la empresa necesita asegurar el 80%, el umbral 0,45 es la alternativa (ver tabla de umbrales vecinos).

### 6.4 KPI de negocio: costo evitado (ingreso protegido)

Se usa `MonthlyCharges` de los clientes de test: el **ingreso mensual en riesgo** es lo que pagan todos los que efectivamente se van; el **ingreso protegible** es lo que pagan los que se van **y** el modelo detecta (verdaderos positivos). Es el ingreso que una campaña de retención podría salvar.

In [ ]:
mc_test = X_test["MonthlyCharges"].values
en_riesgo = mc_test[y_test.values == 1].sum()
detectado = mc_test[(y_test.values == 1) & (pred_final == 1)].sum()
contactados = pred_final.sum()
kpi = pd.Series({
    "Tasa de churn en test": f"{y_test.mean():.1%}",
    "Ingreso mensual en riesgo (todos los que se van)": f"USD {en_riesgo:,.0f}".replace(",", "."),
    "Ingreso mensual de los que se van y el modelo detecta": f"USD {detectado:,.0f}".replace(",", "."),
    "% del ingreso en riesgo que el modelo identifica": f"{detectado/en_riesgo:.1%}",
    "Clientes a contactar": f"{contactados} de {len(y_test)} ({contactados/len(y_test):.1%})",
})
kpi

**Escenario ilustrativo (supuestos explícitos, no datos del dataset).** El ingreso que realmente se retiene depende de cuántos clientes contactados aceptan la oferta, dato que el dataset no tiene y que debería medirse con un piloto. Para dimensionar, se supone un costo de **USD 10 por cliente contactado** y una permanencia de **12 meses** del cliente retenido, y se varía la tasa de éxito de la campaña:

In [ ]:
COSTO_CONTACTO, MESES = 10, 12
escenarios = []
for exito in [0.10, 0.20, 0.30]:
    ingreso_retenido = exito * detectado * MESES
    costo = contactados * COSTO_CONTACTO
    escenarios.append([f"{exito:.0%}", ingreso_retenido, costo, ingreso_retenido - costo])
pd.DataFrame(escenarios, columns=["Tasa de éxito de la campaña", "Ingreso retenido 12 meses (USD)",
                                  "Costo de contactar (USD)", "Beneficio neto (USD)"]).round(0)

**Observación (KPI costo evitado):** contactando al **41,7%** de los clientes, el modelo identifica a quienes concentran **USD 22.542 de los USD 27.215 mensuales en riesgo (82,8%)**. En el escenario ilustrativo, incluso con solo 10% de éxito de la campaña el beneficio neto estimado es positivo (USD 21.180 al año). Estos montos dependen de los supuestos (costo por contacto, permanencia, tasa de éxito), por lo que la recomendación es **medir la tasa de éxito real con un piloto A/B** antes de escalar.

## 7. Interpretación: ¿qué variables usa el modelo?

- **Regresión Logística:** coeficientes convertidos a *odds ratio* (OR = e^coef). OR > 1 aumenta el riesgo; OR < 1 lo reduce. Como las variables numéricas están escaladas a [0, 1], su OR compara el mínimo con el máximo del rango.
- **Random Forest:** importancia por permutación en test (cuánto cae el PR-AUC si se desordena esa variable), más confiable que la importancia interna basada en impureza.

In [ ]:
lr = grid_lr.best_estimator_.named_steps["modelo"]
nombres = [n.split("__", 1)[1] for n in grid_lr.best_estimator_.named_steps["prep"].get_feature_names_out()]
coef = pd.DataFrame({"variable": nombres, "coeficiente": lr.coef_[0]})
coef["odds_ratio"] = np.exp(coef["coeficiente"])
coef = coef.reindex(coef["coeficiente"].abs().sort_values(ascending=False).index)

perm = permutation_importance(grid_rf.best_estimator_, X_test, y_test, scoring="average_precision",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"variable": X_test.columns, "importancia": perm.importances_mean,
                    "desv": perm.importances_std}).sort_values("importancia", ascending=False)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6))
top = coef.head(12).iloc[::-1]
a1.barh(top["variable"], top["coeficiente"], color=[COLOR_SI if v > 0 else COLOR_NO for v in top["coeficiente"]])
a1.axvline(0, color="black", lw=0.8)
a1.set_xlim(top["coeficiente"].min() - 0.9, top["coeficiente"].max() + 0.7)
for yy, (cf, orr) in enumerate(zip(top["coeficiente"], top["odds_ratio"])):
    a1.text(cf + (0.05 if cf > 0 else -0.05), yy, f"OR {orr:.2f}", va="center", ha="left" if cf > 0 else "right", fontsize=9)
a1.set_title("Regresión Logística: 12 coeficientes de mayor magnitude".replace("magnitude", "magnitud"))
a1.set_xlabel("Coeficiente (naranja = aumenta riesgo, azul = reduce)")
top2 = imp.head(12).iloc[::-1]
a2.barh(top2["variable"], top2["importancia"], xerr=top2["desv"], color=C_RF)
a2.set_title("Random Forest: importancia por permutación (test)")
a2.set_xlabel("Caída del PR-AUC al permutar la variable")
fig.suptitle("Gráfico 10. Variables más influyentes", fontweight="bold")
plt.tight_layout(); plt.savefig(IMG + "g10_importancia_variables.png", dpi=130); plt.show()
display(coef.head(12).round(3))
display(imp.head(12).round(4))

**Observación:** ambos modelos coinciden entre sí, con el EDA y con los estudios previos (Aryani y Pratama 2026: tenure, cargos, contrato e internet):
- **Antigüedad (`tenure`)** es la variable más importante en ambos. En la Regresión Logística, con todo lo demás igual, un cliente de 72 meses tiene *odds* de abandono 0,04 veces las de un cliente nuevo (OR 0,04).
- **Contrato:** dos años OR 0,23 y un año OR 0,48 frente a mes a mes. Es la palanca más fuerte que la empresa **sí controla**.
- **Fibra óptica** OR 2,23 (más del doble de *odds* que DSL). **Cheque electrónico** OR 1,51. **Facturación electrónica** OR 1,41.
- **Seguridad online** (OR 0,66) y **soporte técnico** (OR 0,70) reducen el riesgo.
- **Cuidado al leer `TotalCharges` (OR 1,83, positivo):** está muy correlacionada con `tenure` (ρ = 0,89), así que su coeficiente no se interpreta solo: indica que, **a igual antigüedad**, quien ha gastado más tiene más riesgo, coherente con el efecto de los planes caros sin contrato visto en los deciles. El efecto conjunto de la antigüedad sigue siendo fuertemente protector.
- En el Random Forest, las 5 más importantes son `tenure`, fibra óptica, contrato de dos años, `TotalCharges` y contrato de un año.

**Lectura de negocio:** la empresa no puede cambiar la antigüedad, pero sí sabe **cuándo actuar** (primeros meses). Y sí puede actuar sobre el **tipo de contrato, los servicios de protección y el método de pago**.

## 8. Auditoría de sesgo

`gender` y `SeniorCitizen` **no** se usaron para entrenar (Sección 3). Aun así, el modelo podría tratar distinto a esos grupos a través de variables correlacionadas (por ejemplo, fibra óptica o cargo mensual). Se audita de dos formas.

### 8.1 Desempeño por subgrupo
Se compara, para cada grupo: tasa real de churn, % marcado como riesgo, recall (de los que se van, cuántos detecta) y precisión. Un modelo justo en el sentido de *igualdad de oportunidades* debería tener un recall similar entre grupos.

In [ ]:
aud = X_test_all[["gender_Female", "SeniorCitizen"]].copy()
aud["real"], aud["pred"], aud["prob"] = y_test.values, pred_final, prob_test[MODELO_FINAL]
filas = []
for var, etiquetas in [("gender_Female", {0: "Hombre", 1: "Mujer"}), ("SeniorCitizen", {0: "No adulto mayor", 1: "Adulto mayor"})]:
    for val, et in etiquetas.items():
        g = aud[aud[var] == val]
        filas.append([var, et, len(g), g["real"].mean(), g["pred"].mean(),
                      recall_score(g["real"], g["pred"]), precision_score(g["real"], g["pred"])])
tabla_sesgo = pd.DataFrame(filas, columns=["variable", "grupo", "n", "churn real", "% marcado riesgo", "recall", "precisión"])
tabla_sesgo.round(3)

**Observación:**
- **Género:** trato prácticamente igual. Se marca como riesgo a 41,7% de los hombres y 41,6% de las mujeres, con recall 0,796 vs 0,782. El modelo es justo por género.
- **Adulto mayor:** aunque `SeniorCitizen` **no** se usó para entrenar, el modelo marca como riesgo a 67,1% de los adultos mayores vs 36,9% del resto. Lo hace a través de variables relacionadas (fibra, cheque electrónico, contrato mes a mes, Sección 8.2). Esto es coherente con su churn real, que también es mayor (44,1% vs 23,3%), y sus marcas son **más acertadas**, no menos (precisión 0,611 vs 0,466).
- La brecha relevante está en el **recall**: el modelo detecta 92,9% de los adultos mayores que se van, pero solo 73,9% del resto. El grupo con menos cobertura es el de **no** adultos mayores.
- **Implicancia ética:** como la acción es **ofrecer un beneficio** de retención, marcar más a los adultos mayores no los perjudica; sí habría que vigilarlo si el modelo se usara para algo restrictivo. El grupo de adultos mayores en test es pequeño (n = 222), por lo que estas cifras tienen incertidumbre y deben monitorearse con más datos.

### 8.2 Primer vs último decil de riesgo predicho
Siguiendo la sugerencia del docente, se ordena a los clientes de test por probabilidad predicha y se divide en 10 deciles (D1 = menor riesgo, D10 = mayor riesgo). Se compara la tasa real de churn (¿el modelo ordena bien?) y la composición de D1 vs D10 (¿qué perfil concentra el riesgo y si un grupo protegido queda sobrerrepresentado?).

In [ ]:
dec = X_test_all.copy()
dec["churn_real"], dec["prob"] = y_test.values, prob_test[MODELO_FINAL]
dec["decil"] = pd.qcut(dec["prob"].rank(method="first"), 10, labels=range(1, 11)).astype(int)
dec["mes_a_mes"] = 1 - dec["Contract_One year"] - dec["Contract_Two year"]
perfil = dec.groupby("decil").agg(
    prob_media=("prob", "mean"), churn_real=("churn_real", "mean"), tenure_mediana=("tenure", "median"),
    cargo_mensual_mediana=("MonthlyCharges", "median"), mes_a_mes=("mes_a_mes", "mean"),
    fibra=("InternetService_Fiber optic", "mean"), cheque_electronico=("PaymentMethod_Electronic check", "mean"),
    adulto_mayor=("SeniorCitizen", "mean"), mujer=("gender_Female", "mean"))
d1_d10 = perfil.loc[[1, 10]].T
d1_d10.columns = ["D1 (menor riesgo)", "D10 (mayor riesgo)"]
d1_d10["Diferencia D10 − D1"] = d1_d10.iloc[:, 1] - d1_d10.iloc[:, 0]
display(d1_d10.round(3))

lift = perfil["churn_real"] / y_test.mean()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(perfil.index, perfil["churn_real"] * 100, color=[COLOR_SI if d in (1, 10) else "#b0b0b0" for d in perfil.index])
for d, v, l in zip(perfil.index, perfil["churn_real"] * 100, lift):
    ax.text(d, v + 1, f"{v:.0f}%\n(×{l:.1f})", ha="center", fontsize=9)
ax.axhline(y_test.mean() * 100, color="black", ls="--", lw=1, label=f"Tasa global {y_test.mean():.1%}")
ax.set_xticks(range(1, 11)); ax.set_xlabel("Decil de riesgo predicho (1 = menor, 10 = mayor)")
ax.set_ylabel("Tasa real de churn"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(0, perfil["churn_real"].max() * 100 * 1.25); ax.legend(loc="upper left"); ax.grid(axis="x", visible=False)
ax.set_title(f"Gráfico 11. Tasa real de churn por decil de riesgo ({MODELO_FINAL}, test). Entre paréntesis: lift")
plt.tight_layout(); plt.savefig(IMG + "g11_deciles_riesgo.png", dpi=130); plt.show()
print(f"Top 20% de riesgo (D9+D10) concentra el {dec.loc[dec['decil']>=9,'churn_real'].sum()/dec['churn_real'].sum():.1%} de los churn reales de test")

**Observación:**
- **El modelo ordena bien el riesgo:** en el D10 abandona **74,5%** de los clientes (*lift* ×2,8 sobre la tasa global), frente a **1,4%** en el D1. Solo el 20% de mayor riesgo (D9 + D10) concentra el **50% de todos los abandonos** de test: priorizar por decil hace la campaña mucho más eficiente.
- **Perfil del D10 vs D1:** antigüedad mediana de 4 vs 66 meses; cargo mensual mediano de USD 84,05 vs 24,10; 100% vs 0% mes a mes; 97,2% vs 0% fibra; 84,4% vs 2,1% cheque electrónico.
- **Sesgo:** la proporción de mujeres es similar (44,7% en D10 vs 48,9% en D1). La de adultos mayores sube de 5,0% a 31,2% (+26,2 pp) **sin que el modelo use la edad**: el riesgo alto se explica por el plan (fibra, mes a mes, cheque electrónico), donde los adultos mayores están sobrerrepresentados (como se vio en el notebook 1).

## 9. Modelo no supervisado: segmentación de clientes con K-Means

**Objetivo de negocio:** el modelo supervisado dice *quién* está en riesgo; la segmentación dice *qué tipo de cliente es* para diseñar una **oferta distinta por perfil** (no se retiene igual a un cliente nuevo de fibra que a uno antiguo con plan básico).

**Por qué K-Means:** agrupa clientes por similitud en sus características, es eficiente con 7.043 registros y sus centroides son fáciles de describir como "cliente típico" de cada segmento.

**Decisiones de configuración:**
- **Variables:** las mismas 21 del modelo supervisado (servicios, contrato, pago, antigüedad y cargos). **`Churn` no se usa para formar los grupos**: la segmentación descubre perfiles sin conocer la respuesta, y el churn se usa después solo para describir cada segmento. `gender` y `SeniorCitizen` tampoco se usan, por el mismo criterio ético.
- **Escalado:** K-Means usa distancias euclidianas, así que el escalado es obligatorio (sin él, `TotalCharges`, con valores en miles, dominaría la distancia). Se usa el mismo MinMax/Robust del notebook 1, que deja las numéricas en rangos comparables a las variables 0/1.
- **Datos:** todos los clientes (no hay variable objetivo que proteger en este caso).
- **Número de grupos k:** se evalúa k = 2 a 8 con el método del codo (inercia), el coeficiente de silueta y la estabilidad frente a otro algoritmo (clustering jerárquico de Ward, medido con el índice de Rand ajustado, ARI).